In [1]:
import spacy
import pandas as pd
import numpy as np
import time

In [2]:
#nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("xx_ent_wiki_sm")

**Note:** for some reason the English pipeline was not able to extract all the names, not even the English ones (e.g. Cape Town, False Bay, Margate)

In [3]:
df = pd.read_csv("coastal_news_clean.csv")
df.head()

,headlines,publication_date,url,location,event_date,category,date
0,Unprotected and open to plunder,2016-05-29,http://www.iol.co.za/news/south-africa/kwazulu...,Eastern Cape,NaN,illegal fishing,2016-05-29 00:00:00.000000000
1,R303m dedicated towards protecting parts of SA...,NaN,http://traveller24.news24.com/Explore/Green/r3...,"Umgeni River Mouth, near the Blue Lagoon estua...",2016-06-08 00:00:00.000000000,waste,2016-06-08 00:00:00.000000000
2,South Africa: Beach Clean-Up to Mark World Oce...,2016-06-13,http://allafrica.com/stories/201606131641.html,"Umgeni River Mouth, near the Blue Lagoon estua...",2016-06-08 00:00:00.000000000,waste,2016-06-08 00:00:00.000000000
3,Spare a thought on World Oceans Day,2016-06-07,http://www.iol.co.za/capetimes/spare-a-thought...,KwaZulu-Natal South Coast and Seaforth Beach i...,2016-06-08 00:00:00.000000000,sea life,2016-06-08 00:00:00.000000000
4,CSIR’s SeaFAR aiding authorities to protect SA...,NaN,http://www.csir.co.za/news/2016/06/Watching%20...,South Africa,2015-05-01 00:00:00.000000000,development,2015-05-01 00:00:00.000000000


In [4]:
def extract_location(text):
    #Extract location entities from text
    if pd.isna(text):
        return "South Africa"

    doc = nlp(text)

    #Get entities that are locations
    locations = [
        ent.text
        for ent in doc.ents
        if ent.label_ in ['GPE', 'LOC', 'FAC']
    ]

    return locations

In [5]:
# Apply to all articles
df['extracted_location'] = df['location'].apply(extract_location)

In [6]:
# See what we have
for idx, row in df.head(10).iterrows():
    print(f"Article {idx}: {row['headlines']}")
    print(f"Locations: {row['extracted_location']}\n")

Article 0: Unprotected and open to plunder
Locations: ['Eastern Cape']

Article 1: R303m dedicated towards protecting parts of SA's coastline
Locations: ['Umgeni River Mouth', 'Blue Lagoon', 'Durban']

Article 2: South Africa: Beach Clean-Up to Mark World Oceans Day
Locations: ['Umgeni River Mouth', 'Blue Lagoon', 'Durban']

Article 3: Spare a thought on World Oceans Day
Locations: ['Seaforth Beach', 'Simon’s Town']

Article 4: CSIR’s SeaFAR aiding authorities to protect SA’s exclusive economic zone
Locations: ['South Africa']

Article 5: Turning the tide on poaching: Will new policy keep illegal fishing at bay?
Locations: ['South Africa']

Article 6: CSIR technology “sees” vessels even if Automatic Identification Systems are off
Locations: ['South Africa']

Article 7: Navy plays major role in impounding 600 tons of squid and other fish as well as foreign trawlers
Locations: ['East London']

Article 8: Another three Chinese fishing trawlers escorted into a SA port
Locations: ['East Lon

In [7]:
print(df['extracted_location'].value_counts())

extracted_location
[Cape Town]                                                     13
[]                                                              10
[South Africa]                                                   9
[East London]                                                    3
[Port Elizabeth]                                                 3
[Eland's Bay]                                                    3
South Africa                                                     2
[Plettenberg Bay, Knysna, Wilderness]                            2
[Durban]                                                         2
[Elands Bay]                                                     2
[Eastern Cape]                                                   2
[Umgeni River Mouth, Blue Lagoon, Durban]                        2
[Seaforth Beach, Simon’s Town]                                   1
[Durban, Port St Johns, Cape Recife]                             1
[East London Harbour]                      

Extract single location from list/array/string

In [8]:
#from geopy.geocoders import Nominatim
from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter
#from geopandas.tools import geocode

In [9]:
#geolocator = Nominatim(user_agent="coastal_newswatch_v1", timeout=3)
geolocator = ArcGIS(timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

In [15]:
#default = geolocator.geocode("Pretoria, South Africa")
#(default.latitude, default.longitude)

Function to (hopefully) geocode location, handling missing data and multiple locations

In [11]:
def geocode_location(location):
    default = (-25.7460186, 28.1871204) # Default to South Africa center
    # Handle empty/missing
    #print(location)
    if location == [] or location == '' or location == None:
        return  default

    if location == 'South Africa':
        return default
        
    # Handle list of multiple locations (take first one)
    if isinstance(location, list):
        if len(location) == 0:
            return default
            
    location = location[0]  # Use first extracted location
    
    # Handle specific countries
    if location in ["Madagascar", "Mozambique"]:
        search_term = location
    else:
        search_term = f"{location}, South Africa"
    
    try:
        geo_location = geolocator.geocode(search_term)
        # time.sleep(1.5)
        
        if geo_location:
            return (geo_location.latitude, geo_location.longitude)
        else:
            # Geocoding failed, default to South Africa
            return default
            
    except Exception as e:
        print(f"Error geocoding '{location}': {e}")
        return default


In [22]:
'''
for idx, row in df.head(20).iterrows():
    print(f"{row['extracted_location']}")
    print(geocode_location(row['extracted_location']))
'''

'\nfor idx, row in df.head(20).iterrows():\n    print(f"{row[\'extracted_location\']}")\n    print(geocode_location(row[\'extracted_location\']))\n'

In [12]:
#df['coords'] = df['extracted_location'].apply(geocode_location)
df['coordinates'] = df['extracted_location'].apply(geocode_location)

# Split coordinates into separate columns for easier mapping
df[['latitude', 'longitude']] = pd.DataFrame(
    df['coordinates'].tolist(), 
    index=df.index
)

In [13]:
df.head()

,headlines,publication_date,url,location,event_date,category,date,extracted_location,coordinates,latitude,longitude
0,Unprotected and open to plunder,2016-05-29,http://www.iol.co.za/news/south-africa/kwazulu...,Eastern Cape,NaN,illegal fishing,2016-05-29 00:00:00.000000000,[Eastern Cape],"(-32.153662522243, 26.446366021832)",-32.153663,26.446366
1,R303m dedicated towards protecting parts of SA...,NaN,http://traveller24.news24.com/Explore/Green/r3...,"Umgeni River Mouth, near the Blue Lagoon estua...",2016-06-08 00:00:00.000000000,waste,2016-06-08 00:00:00.000000000,"[Umgeni River Mouth, Blue Lagoon, Durban]","(-29.804, 30.9926)",-29.804000,30.992600
2,South Africa: Beach Clean-Up to Mark World Oce...,2016-06-13,http://allafrica.com/stories/201606131641.html,"Umgeni River Mouth, near the Blue Lagoon estua...",2016-06-08 00:00:00.000000000,waste,2016-06-08 00:00:00.000000000,"[Umgeni River Mouth, Blue Lagoon, Durban]","(-29.804, 30.9926)",-29.804000,30.992600
3,Spare a thought on World Oceans Day,2016-06-07,http://www.iol.co.za/capetimes/spare-a-thought...,KwaZulu-Natal South Coast and Seaforth Beach i...,2016-06-08 00:00:00.000000000,sea life,2016-06-08 00:00:00.000000000,"[Seaforth Beach, Simon’s Town]","(-34.2, 18.45)",-34.200000,18.450000
4,CSIR’s SeaFAR aiding authorities to protect SA...,NaN,http://www.csir.co.za/news/2016/06/Watching%20...,South Africa,2015-05-01 00:00:00.000000000,development,2015-05-01 00:00:00.000000000,[South Africa],"(-28.997182288, 25.08504994)",-28.997182,25.085050


In [14]:
# Save the geocoded data
df.to_csv('coastal_news_geocoded.csv', index=False)
print("Data is cleaned and saved")

Data is cleaned and saved


The old geocoding function from back before I realised how incomplete it was

In [ ]:
'''
def geocode_location(location):   
    if location == "Madagascar" or location == "Mozambique":
        geo_location = geolocator.geocode(location)

    else:
        geo_location = geolocator.geocode("%s, South Africa" % location)

    return geo_location.latitude, geo_location.longitude
'''

'\ndef geocode_location(location):   \n    if location == "Madagascar" or location == "Mozambique":\n        geo_location = geolocator.geocode(location)\n\n    else:\n        geo_location = geolocator.geocode("%s, South Africa" % location)\n\n    return geo_location.latitude, geo_location.longitude\n'